In [ ]:
import re
    
    def replace_layer_number(match):
        local_layer_id = int(match.group(1))
        global_layer_id = local_layer_id + total_offset
        return f"layers.{global_layer_id}."
    
    # 匹配 "layers.数字." 的模式
    name = re.sub(r'layers\.(\d+)\.', replace_layer_number, name)

| 部分                              | 含义             |
| ------------------------------- | -------------- |
| `re.sub(pattern, repl, string)` | 正则替换函数         |
| `r'layers\.(\d+)\.'`            | 匹配模式           |
| `replace_layer_number`          | 替换函数（<font color='red'>每次匹配到就调用</font>） |
| `name`                          | 输入的原始参数字符串     |

# 正则表达式详解

In [ ]:
r'layers\.(\d+)\.'

┌─────────────────────────────────────────┐
│  layers  →  字面匹配 "layers"           │
│  \.      →  匹配字面 "." (转义)          │
│  (\d+)   →  捕获组: 匹配一个或多个数字    │
│  \.      →  匹配字面 "." (转义)          │
└─────────────────────────────────────────┘

匹配示例:
  "decoder.layers.0.self_attention..."  →  匹配 "layers.0."
  "decoder.layers.12.mlp..."           →  匹配 "layers.12."
  "embedding.word_embeddings"          →  不匹配（没有 layers.数字.）

# match.group(1) 的含义

In [ ]:
match.group(0)  # 整个匹配的字符串: "layers.0."
match.group(1)  # 第一个捕获组: "0" (括号内的内容)

In [ ]:
输入: "decoder.layers.0.self_attention.query.weight"
       │      │    │ │  │
       │      │    │ │  └── 不匹配
       │      │    │ └──── 匹配 \. 
       │      │    └────── 匹配 (\d+) → group(1) = "0"
       │      └─────────── 匹配 layers\.
       └────────────────── 不匹配（前面部分）
       
匹配结果: "layers.0."  (group(0))
捕获内容: "0"          (group(1))

# 完整执行示例
## 示例 1：GPU 0, VPP Stage 0 (total_offset = 0)

In [ ]:
total_offset = 0

name = "decoder.layers.0.self_attention.query.weight"

# re.sub 找到匹配: "layers.0."
# 调用 replace_layer_number(match)
local_layer_id = int("0") = 0
global_layer_id = 0 + 0 = 0
return "layers.0."

# 替换结果
name = "decoder.layers.0.self_attention.query.weight"  # 不变

## 示例 2：GPU 0, VPP Stage 1 (total_offset = 8)

In [ ]:
total_offset = 8

name = "decoder.layers.0.self_attention.query.weight"

# re.sub 找到匹配: "layers.0."
# 调用 replace_layer_number(match)
local_layer_id = int("0") = 0
global_layer_id = 0 + 8 = 8
return "layers.8."

# 替换结果
name = "decoder.layers.8.self_attention.query.weight"  # 层号变了！

## 示例 3：GPU 1, VPP Stage 1 (total_offset = 24)

In [ ]:
total_offset = 24

name = "decoder.layers.3.mlp.gate_proj.weight"

# re.sub 找到匹配: "layers.3."
local_layer_id = int("3") = 3
global_layer_id = 3 + 24 = 27
return "layers.27."

# 替换结果
name = "decoder.layers.27.mlp.gate_proj.weight"

# 为什么用 re.sub 而不是字符串替换？
## 方案对比

| 方案               | 代码                                       | 问题              |
| ---------------- | ---------------------------------------- | --------------- |
| 字符串替换            | `name.replace("layers.0.", "layers.8.")` | 只能处理特定层号，不通用    |
| `str.replace` 循环 | 遍历所有可能的层号                                | 效率低，代码冗余        |
| **正则替换**         | `re.sub(r'layers\.(\d+)\.', ...)`        | **通用，一次处理所有层号** |

### 正则的优势

In [ ]:
# 一次调用，处理任意层号
name = "decoder.layers.0.self_attention.query.weight"
name = re.sub(r'layers\.(\d+)\.', replace_layer_number, name)
# 结果: "decoder.layers.8.self_attention.query.weight"

name = "decoder.layers.5.mlp.gate_proj.weight"
name = re.sub(r'layers\.(\d+)\.', replace_layer_number, name)
# 结果: "decoder.layers.13.mlp.gate_proj.weight" (如果 offset=8)

# 甚至多层
name = "decoder.layers.0.input_layernorm.weight"
name = re.sub(r'layers\.(\d+)\.', replace_layer_number, name)
# 结果: "decoder.layers.8.input_layernorm.weight"

# 处理多个匹配的情况
## 如果字符串中有多个 layers.数字. 怎么办？

In [ ]:
name = "some.layers.0.nested.layers.1.weight"

# re.sub 默认替换所有匹配
# 第 1 次匹配: "layers.0." → 替换
# 第 2 次匹配: "layers.1." → 替换

# 但通常参数名中只有一个 layers.数字.

##### 实际上，Megatron 的参数名中只有一个 layers.数字.：

In [ ]:
"decoder.layers.0.self_attention.query.weight"
       ↑
       只有一个 layers.0.

# 边界情况处理
## 情况 1：没有匹配

In [ ]:
name = "embedding.word_embeddings.weight"

# re.sub 找不到匹配
# 返回原始字符串，不做修改

## 情况 2：多层号（异常）

In [ ]:
name = "decoder.layers.0.layers.1.weight"  # 异常命名

# 会替换所有匹配:
# "layers.0." → "layers.8."
# "layers.1." → "layers.9."
# 结果: "decoder.layers.8.layers.9.weight"

# 完整流程图

In [ ]:
输入: "decoder.layers.0.self_attention.query.weight"
              ↑
              └── 正则匹配 "layers.0."
              
                    ↓
              ┌─────────────┐
              │ match.group(0) = "layers.0." │
              │ match.group(1) = "0"          │
              └─────────────┘
                    ↓
              replace_layer_number(match)
                    │
                    ├── local_layer_id = int("0") = 0
                    ├── global_layer_id = 0 + total_offset
                    └── return f"layers.{global_layer_id}."
                    
                    ↓
              假设 total_offset = 24
              
              return "layers.24."
              
                    ↓
输出: "decoder.layers.24.self_attention.query.weight"

# 总结

| 元素                                    | 作用                 |
| ------------------------------------- | ------------------ |
| `re.sub`                              | 正则替换，通用处理所有层号      |
| `r'layers\.(\d+)\.'`                  | 匹配 `layers.数字.` 模式 |
| `match.group(1)`                      | 提取捕获的数字（局部层号）      |
| `+ total_offset`                      | 转换为全局层号            |
| `return f"layers.{global_layer_id}."` | 返回替换后的字符串          |

##### 一句话：
- 这段代码用正则表达式自动找到参数名中的层号，通过回调函数加上全局偏移，实现局部层号到全局层号的通用、自动转换。